<a href="https://colab.research.google.com/github/Andresdotdev/PZColab/blob/main/PZ_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# @title 1. Instalar Servidor y Dependencias (Panel Interactivo Limpio)
# @markdown ---
# @markdown ### 🎮 Selección de Versión
Version = "b42 MP - unstable" # @param ["b41", "b42 MP - unstable"]

import os
import subprocess
import re
from IPython.display import clear_output

def mostrar_panel(etapa, progreso_steam=None):
    clear_output(wait=True)
    print("=========================================================")
    print("🚀 INSTALADOR DE SERVIDOR PROJECT ZOMBOID")
    print("=========================================================\n")

    print(f"[{'✅' if etapa >= 1 else '⏳'}] 1. Preparar sistema y dependencias")
    print(f"[{'✅' if etapa >= 2 else '⏳' if etapa == 1 else '  '}] 2. Descargar red Playit.gg")
    print(f"[{'✅' if etapa >= 3 else '⏳' if etapa == 2 else '  '}] 3. Vincular Google Drive")
    print(f"[{'✅' if etapa >= 4 else '⏳' if etapa == 3 else '  '}] 4. Descargar Servidor (SteamCMD)")

    # Mostrar barras de carga múltiples sin saltos
    if progreso_steam and etapa == 3:
        print("\n   📊 PROGRESO DE DESCARGA:")
        for state, pct in progreso_steam.items():
            bar_length = 30
            filled = int(bar_length * pct / 100)
            bar = '█' * filled + '░' * (bar_length - filled)
            print(f"      ► {state:<14} |{bar}| {pct:>5.1f}%")

mostrar_panel(0)

# --- 1. SISTEMA ---
!sudo dpkg --add-architecture i386 > /dev/null 2>&1
!sudo apt update -y > /dev/null 2>&1
!echo steam steam/license note '' | debconf-set-selections
!echo steam steam/question select "I AGREE" | debconf-set-selections
!sudo apt install lib32gcc-s1 lib32stdc++6 steamcmd curl -y > /dev/null 2>&1
# Silenciar la "basura" de actualización inicial de SteamCMD
!/usr/games/steamcmd +quit > /dev/null 2>&1
mostrar_panel(1)

# --- 2. PLAYIT ---
!curl -sL https://github.com/playit-cloud/playit-agent/releases/download/v0.15.26/playit-linux-amd64 -o /usr/local/bin/playit > /dev/null 2>&1
!chmod +x /usr/local/bin/playit
mostrar_panel(2)

# --- 3. GOOGLE DRIVE ---
if not os.path.exists("/content/drive"):
    from google.colab import drive
    drive.mount('/content/drive')
os.makedirs("/content/drive/MyDrive/ZomboidSaves", exist_ok=True)
!rm -rf /root/Zomboid
!ln -s "/content/drive/MyDrive/ZomboidSaves" /root/Zomboid
mostrar_panel(3, {"Iniciando...": 0.0})

# --- 4. STEAMCMD ---
SERVER_PATH = "/content/pzserver"
os.makedirs(SERVER_PATH, exist_ok=True)
steam_args = ['-beta', 'unstable'] if Version == "b42 MP - unstable" else []
cmd = ['/usr/games/steamcmd', '+force_install_dir', SERVER_PATH, '+login', 'anonymous', '+app_update', '380870'] + steam_args + ['+quit']

process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
progreso = {}

for line in process.stdout:
    line_lower = line.lower()
    match = re.search(r'stage\s+\(([^)]+)\)\s+([a-zA-Z]+),\s+progress:\s+([0-9.]+)', line_lower)
    if not match:
        match = re.search(r'update state\s+\(([^)]+)\)\s+([a-zA-Z]+),\s+progress:\s+([0-9.]+)', line_lower)

    if match:
        state = match.group(2).capitalize()
        try:
            percent = float(match.group(3))
        except ValueError:
            percent = 0.0

        if percent > 100.0: percent = 100.0

        progreso[state] = percent
        mostrar_panel(3, progreso)

process.wait()

# --- FINALIZACIÓN ---
if os.path.exists(f"{SERVER_PATH}/ProjectZomboid64"):
    mostrar_panel(4)
    print("\n=========================================================")
    print("✅ ¡FASE 1 COMPLETADA CON ÉXITO! Puedes continuar a la Celda 2.")
    print("=========================================================")
else:
    print("\n⚠️ SteamCMD falló al validar el ejecutable. Intenta correr la celda de nuevo.")

In [ ]:
# @title 2. Configurar Playit.gg Persistente

import os

PLAYIT_DRIVE = "/content/drive/MyDrive/ZomboidSaves/playitgg"
PLAYIT_CONFIG = "/root/.config/playit_gg"

# Matar instancias anteriores
!pkill -f playit 2>/dev/null

# Crear carpeta persistente
os.makedirs(PLAYIT_DRIVE, exist_ok=True)

# Si no existe config local, enlazar Drive
if os.path.exists(PLAYIT_CONFIG):
    !rm -rf {PLAYIT_CONFIG}

!ln -s {PLAYIT_DRIVE} {PLAYIT_CONFIG}

print("✅ Configuración persistente enlazada.")

print("🚀 Iniciando Playit...")
print("⚠️ SOLO la primera vez tendrás que reclamar el túnel.")
print("=" * 50)

!playit

In [ ]:
# @title 3. Iniciar Servidor Project Zomboid
# @markdown ---
server_name = 'PzColab' # @param {type: "string"}
admin_password = 'admin123' # @param {type: "string"}

print("🔥 Iniciando Servidor de Project Zomboid...")
print("⚠️ Nota: El error de 'libjsig.so' es normal, ignóralo.")

# Ejecutamos playit en el fondo
!pkill -f playit 2>/dev/null
!nohup playit > /tmp/playit.log 2>&1 &
print("✅ Túnel encendido en el fondo.")

# Iniciar servidor
!chmod +x /content/pzserver/start-server.sh
!/content/pzserver/start-server.sh -servername {server_name} -adminpassword {admin_password}

🛠️ Script Anti-Abandono para el Navegador
Este script simula que estás haciendo clic en la página de Colab de forma automática cada 10 minutos para engañar al sistema de inactividad.

Pasos para activarlo:

1.   Abre tu cuaderno de Google Colab en el navegador (Chrome, Edge o Firefox).
2.   Presiona la tecla F12 (o clic derecho en cualquier parte de la página y selecciona Inspeccionar).
3.   Ve a la pestaña llamada Consola (Console).
4.   Pega el siguiente código y presiona Enter:
```
function KeepAlive() {
    console.log("Manteniendo servidor activo...");
    // Simula un clic en el botón de conectar o de opciones del sistema
    let connectButton = document.querySelector("#connect") || document.querySelector("colab-connect-button");
    if (connectButton) {
        connectButton.click();
    }
}
setInterval(KeepAlive, 600000); // Se ejecuta automáticamente cada 10 minutos (600,000 ms)
```
Verás un mensaje en la consola cada 10 minutos. Mientras dejes esa pestaña del navegador abierta (aunque minimices la ventana), el servidor no se caerá por inactividad.

In [ ]:
# @title 4. Inyección de Mods por Slots (Compacto con Reporte Visual)
# @markdown ---
# @markdown ### 🌐 Configuración de la Build
Version = "b42 MP - unstable" # @param ["b41", "b42 MP - unstable"]

# @markdown ### 🧹 Control de Historial
Limpiar_Lista_Anterior = False # @param {type:"boolean"}
# @markdown _💡 Activa la casilla de arriba si quieres borrar los mods viejos del servidor y quedarte **solo** con los que escribas abajo._

# @markdown ---
# @markdown ### 📦 Slots de Mods (1 al 10)
# @markdown _Ingresa tus mods aquí. Al ejecutarse, se sumarán a los que ya tienes instalados en el Drive._

# @markdown #### 🔹 MOD 1
workshop_slot_1 = "" # @param {type: "string"}
mod_id_slot_1 = "" # @param {type: "string"}
tipo_slot_1 = "lib" # @param ["lib", "ui", "car", "qol"]

# @markdown #### 🔹 MOD 2
workshop_slot_2 = "" # @param {type: "string"}
mod_id_slot_2 = "" # @param {type: "string"}
tipo_slot_2 = "lib" # @param ["lib", "ui", "car", "qol"]

# @markdown #### 🔹 MOD 3
workshop_slot_3 = "" # @param {type: "string"}
mod_id_slot_3 = "" # @param {type: "string"}
tipo_slot_3 = "ui" # @param ["lib", "ui", "car", "qol"]

# @markdown #### 🔹 MOD 4
workshop_slot_4 = "" # @param {type: "string"}
mod_id_slot_4 = "" # @param {type: "string"}
tipo_slot_4 = "qol" # @param ["lib", "ui", "car", "qol"]

# @markdown #### 🔹 MOD 5
workshop_slot_5 = "" # @param {type: "string"}
mod_id_slot_5 = "" # @param {type: "string"}
tipo_slot_5 = "qol" # @param ["lib", "ui", "car", "qol"]

# @markdown #### 🔹 MOD 6
workshop_slot_6 = "" # @param {type: "string"}
mod_id_slot_6 = "" # @param {type: "string"}
tipo_slot_6 = "qol" # @param ["lib", "ui", "car", "qol"]

# @markdown #### 🔹 MOD 7
workshop_slot_7 = "" # @param {type: "string"}
mod_id_slot_7 = "" # @param {type: "string"}
tipo_slot_7 = "qol" # @param ["lib", "ui", "car", "qol"]

# @markdown #### 🔹 MOD 8
workshop_slot_8 = "" # @param {type: "string"}
mod_id_slot_8 = "" # @param {type: "string"}
tipo_slot_8 = "qol" # @param ["lib", "ui", "car", "qol"]

# @markdown #### 🔹 MOD 9
workshop_slot_9 = "" # @param {type: "string"}
mod_id_slot_9 = "" # @param {type: "string"}
tipo_slot_9 = "qol" # @param ["lib", "ui", "car", "qol"]

# @markdown #### 🔹 MOD 10
workshop_slot_10 = "" # @param {type: "string"}
mod_id_slot_10 = "" # @param {type: "string"}
tipo_slot_10 = "qol" # @param ["lib", "ui", "car", "qol"]

import os, re

SAVES_PATH = '/content/drive/MyDrive/ZomboidSaves'
server_name = 'PzColab'
INI_PATH = f"{SAVES_PATH}/Server/{server_name}.ini"
is_b42 = 'b42' in Version

print(f"⚙️ Procesando inyección compacta para: **{Version.upper()}**\n")

if not os.path.exists(INI_PATH):
    print("❌ ERROR FATAL: No se encontró el archivo INI.")
    print("Inicia el servidor al menos una vez para generar el archivo base.")
else:
    with open(INI_PATH, 'r') as f:
        ini_lines = f.readlines()

    base_mods = []

    # 1. LEER HISTORIAL DEL DRIVE
    if not Limpiar_Lista_Anterior:
        existing_workshop = []
        existing_mods = []

        ini_content = "".join(ini_lines)
        ws_match = re.search(r'^WorkshopItems=(.*)', ini_content, re.MULTILINE)
        if ws_match and ws_match.group(1).strip():
            existing_workshop = [x.strip() for x in ws_match.group(1).split(';') if x.strip()]

        mod_match = re.search(r'^Mods=(.*)', ini_content, re.MULTILINE)
        if mod_match and mod_match.group(1).strip():
            existing_mods = [x.strip().replace('\\', '') for x in mod_match.group(1).split(';') if x.strip()]

        for i in range(min(len(existing_workshop), len(existing_mods))):
            base_mods.append((existing_workshop[i], existing_mods[i], "qol"))
    else:
        print("🧹 Historial limpiado. Se ignoraron los mods antiguos del archivo .ini.")

    # 2. RECOPILAR LOS 10 SLOTS DE LA INTERFAZ
    SLOTS_RAW = [
        (workshop_slot_1, mod_id_slot_1, tipo_slot_1), (workshop_slot_2, mod_id_slot_2, tipo_slot_2),
        (workshop_slot_3, mod_id_slot_3, tipo_slot_3), (workshop_slot_4, mod_id_slot_4, tipo_slot_4),
        (workshop_slot_5, mod_id_slot_5, tipo_slot_5), (workshop_slot_6, mod_id_slot_6, tipo_slot_6),
        (workshop_slot_7, mod_id_slot_7, tipo_slot_7), (workshop_slot_8, mod_id_slot_8, tipo_slot_8),
        (workshop_slot_9, mod_id_slot_9, tipo_slot_9), (workshop_slot_10, mod_id_slot_10, tipo_slot_10)
    ]

    nuevos_mods = []
    for ws, mod_id, tipo in SLOTS_RAW:
        if ws and mod_id and str(ws).strip() and str(mod_id).strip():
            nuevos_mods.append((str(ws).strip(), str(mod_id).strip(), tipo))

    # 3. COMBINAR AMBAS LISTAS EVITANDO DUPLICADOS
    workshop_vistos = set()
    lista_combinada = []

    for m in nuevos_mods:
        if m[0] not in workshop_vistos:
            workshop_vistos.add(m[0])
            lista_combinada.append(m)

    for m in base_mods:
        if m[0] not in workshop_vistos:
            workshop_vistos.add(m[0])
            lista_combinada.append(m)

    # 4. CLASIFICAR SEGÚN EL TIPO DE CARGA
    librerias, interfaces, vehiculos, otros = [], [], [], []
    for ws, mod_id, tipo in lista_combinada:
        entry = (ws, mod_id)
        if tipo == "lib": librerias.append(entry)
        elif tipo == "ui": interfaces.append(entry)
        elif tipo == "car": vehiculos.append(entry)
        else: otros.append(entry)

    MODS_FINALES = librerias + interfaces + vehiculos + otros

    # 5. ESCRIBIR UTILIZANDO REEMPLAZO DIRECTO DE LÍNEAS (Evita errores de escape regex)
    if MODS_FINALES:
        final_ws_ids = [m[0] for m in MODS_FINALES]
        final_mod_ids = [m[1] for m in MODS_FINALES]

        workshop_str = f"WorkshopItems={';'.join(final_ws_ids)}\n"
        mod_str = f"Mods={';'.join([f'\\{m}' if is_b42 else m for m in final_mod_ids])}\n"

        ws_found, mod_found = False, False
        for idx, line in enumerate(ini_lines):
            if line.startswith("WorkshopItems="):
                ini_lines[idx] = workshop_str
                ws_found = True
            elif line.startswith("Mods="):
                ini_lines[idx] = mod_str
                mod_found = True

        if not ws_found: ini_lines.append(workshop_str)
        if not mod_found: ini_lines.append(mod_str)

        with open(INI_PATH, 'w') as f:
            f.writelines(ini_lines)

        # 6. REPORTE VISUAL DETALLADO POR CONSOLA
        print("=========================================================")
        print(f"📋 LISTA ACTUAL DE MODS EN EL SERVIDOR (Total: {len(MODS_FINALES)})")
        print("=========================================================")

        if librerias:
            print("\n📚 [LIBRERÍAS / CORE] (Cargan Primero):")
            for ws, mod_id in librerias: print(f"   ➡️ Mod ID: {mod_id} | Workshop ID: {ws}")
        if interfaces:
            print("\n🖥️ [INTERFAZ / UI]:")
            for ws, mod_id in interfaces: print(f"   ➡️ Mod ID: {mod_id} | Workshop ID: {ws}")
        if vehiculos:
            print("\n🚗 [VEHÍCULOS]:")
            for ws, mod_id in vehiculos: print(f"   ➡️ Mod ID: {mod_id} | Workshop ID: {ws}")
        if otros:
            print("\n⚙️ [UTILIDADES / QoL / HISTORIAL]:")
            for ws, mod_id in otros: print(f"   ➡️ Mod ID: {mod_id} | Workshop ID: {ws}")

        print("\n=========================================================")
        print("✅ Archivo .ini actualizado con éxito.")
    else:
        print("⚠️ No hay mods activos actualmente en el servidor.")

In [ ]:
# @title 🔍 4.1 Inspector y Diagnóstico Avanzado de Servidor (Ultra-Robusto)
# @markdown _Muestra tus mods activos y analiza los logs agrupando errores por mod e identificando culpables reales._

import os, re

SAVES_PATH = '/content/drive/MyDrive/ZomboidSaves'
server_name = 'PzColab'
INI_PATH = f"{SAVES_PATH}/Server/{server_name}.ini"

# --- PARTE 1: LECTURA DE MODS ---
print("=========================================================")
print("👁️  MODS CONFIGURADOS EN EL SERVIDOR")
print("=========================================================")

if not os.path.exists(INI_PATH):
    print("❌ No se encontró ningún archivo de configuración .ini.")
else:
    with open(INI_PATH, 'r') as f:
        content = f.read()

    ws_match = re.search(r'^WorkshopItems=(.*)', content, re.MULTILINE)
    mod_match = re.search(r'^Mods=(.*)', content, re.MULTILINE)

    ws_list = [x.strip() for x in ws_match.group(1).split(';') if x.strip()] if ws_match else []
    mod_list = [x.strip().replace('\\', '') for x in mod_match.group(1).split(';') if x.strip()] if mod_match else []
    total_mods = min(len(ws_list), len(mod_list))

    if total_mods > 0:
        for i in range(total_mods):
            nombre = mod_list[i]
            tag = "📦 Mod"
            if "lib" in nombre.lower() or "tsar" in nombre.lower(): tag = "📚 Lib"
            elif "ui" in nombre.lower(): tag = "🖥️ UI"
            elif "car" in nombre.lower() or "vehicle" in nombre.lower(): tag = "🚗 Car"
            print(f"[{i+1:>2}] {tag} ➡️ ID: {nombre:<25} | Workshop: {ws_list[i]}")
    else:
        print("⚠️ El archivo .ini no tiene mods configurados.")
print("=========================================================\n")


# --- PARTE 2: DIAGNÓSTICO INTELIGENTE ---
print("🔍 INICIANDO ESCANEO AVANZADO DE LOGS...")
print("=========================================================")

log_files = []
for root, dirs, files in os.walk(SAVES_PATH):
    for file in files:
        if file.endswith('.txt') and ('debuglog' in file.lower() or 'console' in file.lower()):
            log_files.append(os.path.join(root, file))

if not log_files:
    print("ℹ️ No se encontraron archivos de registro activos.")
else:
    ULTIMO_LOG = max(log_files, key=os.path.getmtime)
    print(f"📖 Analizando incidencias en: MIdrive/{ULTIMO_LOG.replace('/content/drive/MyDrive/', '')}\n")

    with open(ULTIMO_LOG, 'r', encoding='utf-8', errors='ignore') as f:
        log_lines = f.readlines()

    fallos_criticos = []
    alertas_esteticas = []
    errores_steam = []

    # Diccionario para contar qué mods están dando más guerra
    mods_culpables = {}

    for idx, line in enumerate(log_lines):
        line_lower = line.lower()
        num_linea = idx + 1

        # 1. DETECTAR CRASHES O ERRORES LUA
        if "lua error" in line_lower or "stack trace" in line_lower or "call stack" in line_lower:
            # Rastrear contexto (buscar el nombre del mod culpable en las 4 líneas cercanas)
            contexto = "Desconocido (Script interno)"
            for k in range(max(0, idx-2), min(len(log_lines), idx+3)):
                match_mod = re.search(r'(media/lua/[^\s]+|mods/([^/\s]+))', log_lines[k])
                if match_mod:
                    contexto = match_mod.group(1)
                    break

            fallos_criticos.append((num_linea, line.strip(), contexto))
            mods_culpables[contexto] = mods_culpables.get(contexto, 0) + 1

        # 2. DETECTAR FALLOS DE STEAM WORKSHOP
        elif "workshop" in line_lower and ("fail" in line_lower or "error" in line_lower or "rejected" in line_lower):
            errores_steam.append(f"[Línea {num_linea}] 🌐 Fallo Steam ➡️ {line.strip()}")

        # 3. FILTRAR ALERTAS MENORES (Evita alarmar por sonidos o vallas rotas)
        elif "missing" in line_lower and ("thumpsound" in line_lower or "tile" in line_lower or "media/sound" in line_lower):
            alertas_esteticas.append(f"[Línea {num_linea}] 📝 Detalle ➡️ {line.strip()[:90]}...")

    # --- DESPLIEGUE DEL REPORTE RESUMIDO ---
    if fallos_criticos or errores_steam or alertas_esteticas:

        if fallos_criticos:
            print(f"🔴 Errores de Lua/Crashes Detectados: {len(fallos_criticos)}")
            print("👑 MODS O ARCHIVOS MÁS INESTABLES:")
            for mod, count in sorted(mods_culpables.items(), key=lambda x: x[1], reverse=True)[:3]:
                print(f"   ⚠️ -> '{mod}' generó {count} alertas en este arranque.")
            print("\n📌 Muestra de las primeras líneas de error:")
            for num, err, ctx in fallos_criticos[:4]:
                print(f"   [Línea {num}] Script: {ctx}")
            print("-" * 60)

        if errores_steam:
            print(f"\n🌐 Problemas con Steam Workshop: {len(errores_steam)}")
            for err in errores_steam[:3]: print(f"   {err}")
            print("-" * 60)

        if alertas_esteticas:
            print(f"\n📝 Alertas Menores o Estéticas (No rompen el servidor): {len(alertas_esteticas)}")
            print("   💡 _Nota: Son sonidos faltantes o vallas del mapa original. Ignorables._")
            for err in alertas_esteticas[:3]: print(f"   {err}")
            print("-" * 60)

        print("\n🛠️ DIAGNÓSTICO FINAL:")
        if fallos_criticos:
            print("   El servidor inició, pero hay mods con scripts obsoletos. Si notas lag visual o items invisibles,")
            print("   revisa los mods indicados en el top de inestabilidad.")
        else:
            print("   ¡Estable! El servidor no registra problemas de programación críticos en los mods.")
    else:
        print("✅ ¡SISTEMA 100% LIMPIO! Logs impecables y listos para jugar.")

print("=========================================================")